# SEA-Net - data analysis

A thin, read-only notebook. All the figures and tables are built by **`seanet/report.py`** (the same
code `python main.py report` runs), so the plotting lives in one place, not here.

This notebook just calls it and shows the results. It trains nothing; run it any time to see
whatever has finished so far.

**One model = one encoder + one pooling head = one folder.** Every model writes into its own
`results/SEA_NET/<encoder>_<pooling>/`, so the cells below loop over the models that have results.

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root (the folder that contains the "seanet" package) and make it importable.
root = Path.cwd()
while root != root.parent and not (root / "seanet" / "__init__.py").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

from seanet import report
from seanet import results as R
from IPython.display import Image, display
print("repo root:", root)
print("models with results:", R.discover_models())

## 1. Generate everything

This refreshes every model's comparison + summary and draws all the figures - the same call as
`python main.py report`.

In [ ]:
out = report.generate_report(verbose=True)

## 2. Which model wins?

The cross-model ranking over the **85 datasets MILLET published** - the only ones where a fair
one-to-one comparison exists. MILLET's own means are in the `mean_*_millet` columns, so a model
beats the baseline when its `mean_acc_ours` clears `mean_acc_millet`.

In [ ]:
cross = R.compare_models(verbose=False)
display(cross[["model", "mean_acc_ours", "mean_acc_millet", "acc_win_tie_loss",
               "mean_loss_ours", "mean_loss_millet", "mean_aopcr_ours", "mean_aopcr_millet"]]
       if len(cross) else "No model has results yet - run: python main.py train --model seanet")

# our new pooling heads in orange, MILLET's own bar in red
shared = report.SHARED_FIGURES_DIR
for name in ["model_comparison.png", "data_summary.png"]:
    path = os.path.join(shared, name)
    if os.path.exists(path):
        display(Image(filename=path))

## 3. Each model's figures

For every model: its own spread, then the comparison to MILLET. In the accuracy and AOPCR scatters a
point **above** the red y=x line means we win; in the loss scatter **below** the line wins (lower
loss is better). The gold star marks both means.

In [ ]:
for model_id in R.discover_models():
    print("=" * 70)
    print(model_id)
    print("=" * 70)
    figdir = R.figures_dir(model_id)
    for name in ["results.png", "means.png", "acc_scatter.png", "loss_scatter.png",
                 "aopcr_scatter.png", "win_tie_loss.png", "acc_diff.png"]:
        path = os.path.join(figdir, name)
        if os.path.exists(path):
            display(Image(filename=path))

## 4. The underlying tables

The data behind the figures: one model's per-dataset results, and its comparison to MILLET (only the
85 datasets that overlap the paper have a MILLET number; the other ~43 are marked `no_baseline`).

In [ ]:
models = R.discover_models()
if models:
    model_id = models[0]                      # change this to look at a different model
    print("showing:", model_id)

    results = R.load_results(model_id)
    print("datasets with results:", len(results))
    display(results[["dataset", "test_acc", "test_loss", "test_aopcr", "test_ndcg",
                     "train_time_s", "run_datetime"]].head())

    cmp = R.build_comparison(model_id, verbose=False)
    overlap = R.overlap_rows(cmp)
    print(f"datasets with a MILLET baseline: {len(overlap)} of {len(cmp)}")
    display(overlap[["dataset", "ours_acc", "millet_acc", "acc_outcome",
                     "ours_loss", "millet_loss", "loss_outcome",
                     "ours_aopcr", "millet_aopcr", "aopcr_outcome"]].head(10))

    print("
headline summary:")
    for key, value in R.summarise_model(model_id).items():
        print(f"  {key:24s}: {value}")
else:
    print("No model has results yet - run: python main.py train --model seanet")

## Notes

- **The fair comparison is the 85 datasets MILLET published.** The `overall_mean_*` numbers cover
  every UCR dataset we trained; MILLET never reported those, so there is nothing to compare them to.
- **Accuracy** vs MILLET: the accuracy scatter, the win/tie/loss bars, and the per-dataset gap.
- **Loss**: lower is better, so in `loss_scatter.png` the winning side is *below* the line.
- **AOPCR** (interpretation faithfulness): the AOPCR scatter. This has been SEA-Net's weak spot -
  it is the main reason the new pooling heads exist.
- **NDCG@n** only exists for WebTraffic (the only dataset with per-timestep ground truth); see each
  model's `summary.csv`.
- To regenerate outside the notebook: `python main.py report`.
- Re-run after more of a sweep finishes to refresh.